In [ ]:
!pip uninstall -y torch torchvision torchaudio triton
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 triton==3.1.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q datasets pandas tqdm pyarrow

In [ ]:
import torch
import pandas as pd
import traceback
import importlib.util
import uuid

from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset


DATASET_NAME = "GPUMODE/KernelBook"
SPLIT = "train"
OUTPUT_FILE = "kernelbook_benchmarked.parquet"

DEVICE = "cuda"

KERNELBOOK_LIMIT = 18162
WARMUP_RUNS = 3
BENCH_RUNS = 10
DEBUG_ROWS = 10

TEMP_MODULE_DIR = Path("temp_kernelbook_modules")
TEMP_MODULE_DIR.mkdir(exist_ok=True)


def sync_cuda():
    torch.cuda.synchronize()


def move_to_cuda(obj):
    if torch.is_tensor(obj):
        return obj.to(DEVICE)

    if isinstance(obj, list):
        return [move_to_cuda(x) for x in obj]

    if isinstance(obj, tuple):
        return tuple(move_to_cuda(x) for x in obj)

    if isinstance(obj, dict):
        return {k: move_to_cuda(v) for k, v in obj.items()}

    return obj


def normalize_inputs(inputs):
    if isinstance(inputs, tuple):
        return inputs

    if isinstance(inputs, list):
        return tuple(inputs)

    return (inputs,)


def flatten_tensors(obj):
    tensors = []

    if torch.is_tensor(obj):
        tensors.append(obj)

    elif isinstance(obj, (list, tuple)):
        for item in obj:
            tensors.extend(flatten_tensors(item))

    elif isinstance(obj, dict):
        for item in obj.values():
            tensors.extend(flatten_tensors(item))

    return tensors


def outputs_match(output_a, output_b, atol=1e-4, rtol=1e-4):
    tensors_a = flatten_tensors(output_a)
    tensors_b = flatten_tensors(output_b)

    if len(tensors_a) != len(tensors_b):
        return False

    for a, b in zip(tensors_a, tensors_b):
        if a.shape != b.shape:
            return False

        if not torch.allclose(a, b, atol=atol, rtol=rtol):
            return False

    return True


def benchmark_function(fn, inputs):
    for _ in range(WARMUP_RUNS):
        fn(*inputs)

    sync_cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    for _ in range(BENCH_RUNS):
        fn(*inputs)

    end.record()
    sync_cuda()

    return start.elapsed_time(end) / BENCH_RUNS


def parse_init_inputs(namespace):
    if "get_init_inputs" not in namespace:
        return [], {}

    init_data = namespace["get_init_inputs"]()

    if (
        isinstance(init_data, list)
        and len(init_data) == 2
        and isinstance(init_data[0], list)
        and isinstance(init_data[1], dict)
    ):
        return init_data[0], init_data[1]

    if isinstance(init_data, tuple):
        return list(init_data), {}

    if isinstance(init_data, list):
        return init_data, {}

    return [init_data], {}


def build_pytorch_model_and_inputs(python_code, entry_point):
    namespace = {}
    exec(python_code, namespace)

    if entry_point not in namespace:
        raise RuntimeError(f"No existe entry_point: {entry_point}")

    if "get_inputs" not in namespace:
        raise RuntimeError("No existe get_inputs()")

    raw_inputs = namespace["get_inputs"]()
    inputs = normalize_inputs(raw_inputs)
    inputs = move_to_cuda(inputs)

    entry = namespace[entry_point]

    if isinstance(entry, type):
        init_args, init_kwargs = parse_init_inputs(namespace)

        model = entry(*init_args, **init_kwargs)
        model = model.to(DEVICE)
        model.eval()

        def pytorch_fn(*args):
            with torch.no_grad():
                return model(*args)

        return pytorch_fn, inputs, model, init_args, init_kwargs

    if callable(entry):
        def pytorch_fn(*args):
            with torch.no_grad():
                return entry(*args)

        return pytorch_fn, inputs, None, [], {}

    raise RuntimeError("El entry_point de PyTorch no es callable")


def load_triton_module(triton_code):
    module_name = f"triton_module_{uuid.uuid4().hex}"
    module_path = TEMP_MODULE_DIR / f"{module_name}.py"

    module_path.write_text(triton_code, encoding="utf-8")

    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


def build_triton_function(triton_code, entry_point, init_args, init_kwargs, pytorch_model):
    module = load_triton_module(triton_code)

    new_class_name = f"{entry_point}New"

    if hasattr(module, new_class_name):
        triton_class = getattr(module, new_class_name)

        triton_model = triton_class(*init_args, **init_kwargs)
        triton_model = triton_model.to(DEVICE)
        triton_model.eval()

        if pytorch_model is not None:
            triton_model.load_state_dict(pytorch_model.state_dict(), strict=False)

        def triton_fn(*args):
            with torch.no_grad():
                return triton_model(*args)

        return triton_fn, "model_new"

    if hasattr(module, "call"):
        call_fn = module.call

        def triton_fn(*args):
            with torch.no_grad():
                return call_fn(list(args))

        return triton_fn, "call"

    raise RuntimeError("No existe clase New ni función call en triton_code")


def process_row(row, debug=False):
    try:
        entry_point = row["entry_point"]

        pytorch_fn, inputs, pytorch_model, init_args, init_kwargs = build_pytorch_model_and_inputs(
            python_code=row["python_code"],
            entry_point=entry_point,
        )

        triton_fn, triton_mode = build_triton_function(
            triton_code=row["triton_code"],
            entry_point=entry_point,
            init_args=init_args,
            init_kwargs=init_kwargs,
            pytorch_model=pytorch_model,
        )

        pytorch_output = pytorch_fn(*inputs)
        sync_cuda()

        triton_output = triton_fn(*inputs)
        sync_cuda()

        if debug:
            print("\n" + "=" * 80)
            print("ENTRY POINT:", entry_point)
            print("UUID:", row.get("uuid"))
            print("Triton mode:", triton_mode)
            print("Inputs:", len(inputs))
            print("PyTorch output type:", type(pytorch_output))
            print("Triton output type:", type(triton_output))
            print("PyTorch tensors:", len(flatten_tensors(pytorch_output)))
            print("Triton tensors:", len(flatten_tensors(triton_output)))

        if not outputs_match(pytorch_output, triton_output):
            if debug:
                print("RESULTADO: outputs NO coinciden")
            return None

        pytorch_ms = benchmark_function(pytorch_fn, inputs)
        triton_ms = benchmark_function(triton_fn, inputs)

        new_row = dict(row)
        new_row["pytorch_avg_ms"] = float(pytorch_ms)
        new_row["triton_avg_ms"] = float(triton_ms)
        new_row["triton_is_faster"] = bool(triton_ms < pytorch_ms)

        if debug:
            print("RESULTADO: fila válida")
            print("PyTorch ms:", pytorch_ms)
            print("Triton ms:", triton_ms)
            print("Triton es más rápido:", triton_ms < pytorch_ms)

        return new_row

    except Exception as e:
        if debug:
            print("\n" + "=" * 80)
            print("ERROR EN FILA")
            print("Entry point:", row.get("entry_point"))
            print("UUID:", row.get("uuid"))
            print("Error:", repr(e))
            traceback.print_exc(limit=2)

        return None


def main():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Activa GPU en Colab: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU"
        )

    dataset = load_dataset(DATASET_NAME, split=SPLIT)

    if KERNELBOOK_LIMIT is not None:
        dataset = dataset.select(range(KERNELBOOK_LIMIT))

    valid_rows = []
    removed_rows = 0
    processed_rows = 0

    for row in tqdm(dataset, desc="Benchmarking KernelBook"):
        debug = processed_rows < DEBUG_ROWS

        result = process_row(row, debug=debug)

        processed_rows += 1

        if result is None:
            removed_rows += 1
        else:
            valid_rows.append(result)

    df = pd.DataFrame(valid_rows)
    df.to_parquet(OUTPUT_FILE, index=False)

    print("\nListo")
    print("Filas procesadas:", processed_rows)
    print("Filas válidas:", len(valid_rows))
    print("Filas eliminadas:", removed_rows)
    print("Archivo guardado:", OUTPUT_FILE)

    if len(valid_rows) > 0:
        display(
            df[
                [
                    "entry_point",
                    "pytorch_avg_ms",
                    "triton_avg_ms",
                    "triton_is_faster",
                ]
            ]
        )


main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.71k [00:00<?, ?B/s]

dataset_permissive.parquet:   0%|          | 0.00/87.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18162 [00:00<?, ? examples/s]

Benchmarking KernelBook:   0%|          | 1/18162 [00:05<25:27:08,  5.05s/it]

...

Benchmarking KernelBook:   0%|          | 10/18162 [00:09<2:41:16,  1.88it/s]


ENTRY POINT: GCN
UUID: 10
Triton mode: model_new
Inputs: 2
PyTorch output type: <class 'torch.Tensor'>
Triton output type: <class 'torch.Tensor'>
PyTorch tensors: 1
Triton tensors: 1
RESULTADO: outputs NO coinciden


Benchmarking KernelBook:   0%|          | 64/18162 [00:25<1:26:00,  3.51it/s]<string>:71: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
/content/temp_kernelbook_modules/triton_module_89eb43d488c04a7aafe3d548f1737942.py:212: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.weight)
Benchmarking KernelBook:   2%|▏         | 307/18162 [01:35<1:44:49,  2.84it/s]<string>:22: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
<string>:22: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
Benchmarking KernelBook:   2%|▏         | 353/18162 [01:47<1:24:54,  3.50it/s]<string>:27: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the follow

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_27052/90831209.py", line 335, in <cell line: 0>
    main()
  File "/tmp/ipykernel_27052/90831209.py", line 304, in main
    result = process_row(row, debug=debug)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_27052/90831209.py", line 228, in process_row
    triton_fn, triton_mode = build_triton_function(
                             ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_27052/90831209.py", line 187, in build_triton_function
    module = load_triton_module(triton_code)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_27052/90831209.py", line 181, in load_triton_module
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, i

TypeError: object of type 'NoneType' has no len()